In [110]:
import numpy as np
import random
import math 
seed1 = 0

# Tworzymy generatory liczb losowych z różnymi seedami
generator_uni_i = np.random.default_rng(seed1)
generator_uni_r = np.random.default_rng(seed1 + 100)
generator_norm = np.random.default_rng(seed1 + 200)
generator_cachy = np.random.default_rng(seed1 + 300)
generator_uni_i_2 = np.random.default_rng(seed1 + 400)

# Funkcje losowe

def IntRandom(target: int) -> int:
    if target == 0:
        return 0
    # modulo jak w C++
    return int(generator_uni_i.integers(0, 32769) % target)

def Random(minimal: float, maximal: float) -> float:
    return generator_uni_r.uniform(0.0, 1.0) * (maximal - minimal) + minimal

def NormRand(mu: float, sigma: float) -> float:
    return generator_norm.normal(0.0, 1.0) * sigma + mu

def CachyRand(mu: float, sigma: float) -> float:
    return generator_cachy.standard_cauchy() * sigma + mu


# qSort1 - szybkie sortowanie tablicy liczb zmiennoprzecinkowych
# sorted_list = sorted(your_list)
# lub metody list.sort()
#
# Jeśli masz numpy array:
# sorted_array = np.sort(your_array)

# qSort2int - sortowanie dwóch tablic, gdzie druga jest "dopasowana" do pierwszej (sortowanie po Mass, przesuwając elementy w Mass2)
# np.
# pairs = list(zip(Mass, Mass2))
# pairs.sort(key=lambda x: x[0])
# Mass_sorted, Mass2_sorted = zip(*pairs)


# -------------------------------------------------

# void cec21_test_func(double *, double *,int,int,int);
def test_func(X, dim):
    X = np.array(X)
    y = np.sum(X[:dim]**2)
    return y  

import numpy as np
import random

OShift = None
M = None
y = None
z = None
x_bound = None
ini_flag = 0
n_flag = 0
func_flag = 0
SS = None
stepsFEval = np.zeros(16)
ResultsArray = np.zeros((10, 30, 16))
LastFEcount = 0
NFEval = 0
maxFes = 10000
tempF = np.zeros(1)
fopt = 0.0
buffer = ""
globalbest = 0.0
globalbestinit = False
initfinished = False
FitTemp3 = []
dimensionality=10


NMBR_OF_RUNS=30
NUM_OF_DUMPS=16



MIN_ERROR=1e-8
MAX_INT = 32768
# def IntRandom(target):
#     if target == 0:
#         return 0
#     return random.randint(0, 32768) % target
# random.randint(0, MAX_INT) % target
# MAX_INT = 32768


# def Random(minimal, maximal):
#     return random.uniform(minimal, maximal)
# random.uniform(minimal, maximal)


def GenerateNextRandUnif(num, Range, Rands, Prohib=None):
    for _ in range(25):
        candidate = IntRandom(Range)
        if candidate not in Rands[:num]:
            Rands[num] = candidate
            break
    else:
        # Po 25 próbach przyjmij ostatnią wygenerowaną
        Rands[num] = candidate

def GenerateNextRandUnifOnlyArch(num, Range, Range2, Rands, Prohib=None):
    for _ in range(25):
        candidate = IntRandom(Range2) + Range
        if candidate not in Rands[:num]:
            Rands[num] = candidate
            break
    else:
        Rands[num] = candidate

def CheckGenerated(num, Rands, Prohib):
    if Rands[num] == Prohib:
        return False
    for j in range(num):
        if Rands[j] == Rands[num]:
            return False
    return True

# def SaveBestValues(funcN, RunN, newbestfit):

def FindLimits(Ind, Parent, CurNVars, CurLeft, CurRight):
    for j in range(CurNVars):
        if Ind[j] < CurLeft:
            Ind[j] = Random(CurLeft, CurRight)
        if Ind[j] > CurRight:
            Ind[j] = Random(CurLeft, CurRight)


In [111]:
class Optimizer():
    def __init__(self, newNInds, newNVars, new_func_num, 
                 newRunN, new_bench_num, NewMemSize, NewArchSizeParam):
        
        self.FitNotCalculated = True # bool
        self.MemorySize = NewMemSize # Int 
        self.MemoryIter = 0 # Int 
        self.SuccessFilled = 0 # Int 
        self.MemoryCurrentIndex = None # Int 
        self.NVars = newNVars # Int 
        self.NInds = newNInds # Int 
        self.popSize = self.NInds
        self.NIndsMax = self.NInds # Int 
        self.NIndsMin = 4 # Int 
        self.besti = 0 # Int 
        self.func_num = new_func_num  # Int 
        self.bench_num = new_bench_num # Int 
        self.RunN = newRunN # Int 
        self.Generation = 0 # Int 
        self.CurrentArchiveSize = 0 # Int 
        self.F = 0.2 # double 
        self.Cr = 0.2 # double 
        self.bestfit = float('inf') # double 
        self.ArchiveSizeParam = NewArchSizeParam # double       
        self.ArchiveSize = self.NIndsMax*self.ArchiveSizeParam # Int 
        self.Int_ArchiveSizeParam = math.ceil(self.ArchiveSizeParam) # Int  
        self.Right = 100 # double 
        self.Left = -100 # double 
        self.Rands = np.zeros(self.NIndsMax) # int* 
        self.Indexes = np.zeros(self.NIndsMax) # int* 
        self.BackIndexes = np.zeros(self.NIndsMax) # int*

        self.Weights = np.zeros(self.NIndsMax) # double* 
        self.Donor = np.zeros(self.NVars) # double* 
        self.Trial = np.zeros(self.NVars)  # double* 
        self.FitMass = np.zeros(self.NIndsMax) # double* 
        self.FitMassTemp = np.zeros(self.NIndsMax) # double* 
        self.FitMassCopy = np.zeros(self.NIndsMax) # double* 
        self.BestInd = np.zeros(self.NVars) # double* 
        self.tempSuccessCr = np.zeros(self.NIndsMax) # double* 
        self.tempSuccessF = np.zeros(self.NIndsMax) # double* 
        self.FGenerated = np.zeros(self.NIndsMax)  # double* 
        self.CrGenerated = np.zeros(self.NIndsMax) # double* 
        self.MemoryCr = np.full(self.MemorySize, 0.2) # double* 
        self.MemoryF = np.full(self.MemorySize, 0.2) # double* 
        self.FitDelta = np.zeros(self.NIndsMax) # double* 
        self.ArchUsages = np.zeros(self.NIndsMax)  # double* 
        
        self.Popul = np.zeros((self.NIndsMax, self.NVars)) # double** 
        for i in range(self.NIndsMax):
            for j in range(self.NVars):
                self.Popul[i][j] = Random(self.Left, self.Right)        

        self.PopulTemp = np.zeros((self.NIndsMax, self.NVars)) # double** 
        self.Archive = np.zeros((self.NIndsMax*self.Int_ArchiveSizeParam, self.NVars)) # double** 

        for steps_k in range (15):
            stepsFEval[steps_k] = pow(dimensionality,steps_k/5.0-3.0)
        stepsFEval[15] = 1.0

    def SaveSuccessCrF(self, Cr, F, FitD):
        self.tempSuccessCr[self.SuccessFilled] = Cr
        self.tempSuccessF[self.SuccessFilled] = F
        self.FitDelta[self.SuccessFilled] = FitD
        self.SuccessFilled += 1


    def MeanWL_general(self, Vector, TempWeights, Size, g_p, g_m):
        SumWeight = 0
        SumSquare = 0
        Sum = 0
        for i in range(self.SuccessFilled):
            SumWeight += TempWeights[i]

        for i in range(self.SuccessFilled):
            self.Weights[i] = TempWeights[i]/SumWeight

        for i in range(self.SuccessFilled):
            SumSquare += self.Weights[i]*pow(Vector[i],g_p)

        for i in range(self.SuccessFilled):
            Sum += self.Weights[i]*pow(Vector[i],g_p-g_m)

        if(abs(Sum) > 0.000001):
            return SumSquare/Sum
        else:
            return 0.5


    def UpdateMemoryCrF(self):
        if(self.SuccessFilled != 0):
            self.MemoryCr[self.MemoryIter] = self.MeanWL_general(self.tempSuccessCr,self.FitDelta,self.SuccessFilled,2,1)
            self.MemoryF[self.MemoryIter] = self.MeanWL_general(self.tempSuccessF, self.FitDelta,self.SuccessFilled,2,1)
            self.MemoryIter += 1
            if(self.MemoryIter >= self.MemorySize):
                self.MemoryIter = 0
        else:
            self.MemoryF[self.MemoryIter] = 0.5
            self.MemoryCr[self.MemoryIter] = 0.5


    def CopyToArchive(self, RefusedParent, RefusedFitness):
        if(self.CurrentArchiveSize < self.ArchiveSize):
            for i in range(self.NVars):
                self.Archive[self.CurrentArchiveSize][i] = RefusedParent[i]
            self.CurrentArchiveSize += 1

        elif(self.ArchiveSize > 0):
            RandomNum = int(random.randint(0, MAX_INT) % self.ArchiveSize)
            for i in range(self.NVars):
                self.Archive[RandomNum][i] = RefusedParent[i]

    def FindNSaveBest(self, init, ChosenOne):
        global globalbest, globalbestinit

        if(self.FitMass[ChosenOne] <= self.bestfit or init):
            self.bestfit = self.FitMass[ChosenOne]
            self.besti = ChosenOne
            for j in range(self.NVars):
                self.BestInd[j] = self.Popul[self.besti][j]

        if(self.bestfit < globalbest):
            globalbest = self.bestfit
    

    def RemoveWorst(self, NInds, NewNInds):       
        PointsToRemove = self.NInds - NewNInds
        for L in range (PointsToRemove):
            self.WorstFit = self.FitMass[0]
            self.WorstNum = 0
            for i in range(NInds):
                if(self.FitMass[i] > self.WorstFit):
                    self.WorstFit = self.FitMass[i]
                    self.WorstNum = i

            for i in range(self.WorstNum, self.NInds - 1):
                for j in range(self.NVars):
                    self.Popul[i][j] = self.Popul[i+1][j]
                self.FitMass[i] = self.FitMass[i+1]

    
    def GetValue(self, index, NInds, j):
        idx = int(index)
        if(idx < NInds):
            return self.Popul[idx][j]
        return self.Archive[idx-NInds][j]
    


#------------------------------------------------

    def MainCycle(self):
        global globalbest, globalbestinit

        ArchSuccess = 0.0
        NoArchSuccess = 0.0
        NArchUsages = 0.0
        ArchProbs = 0.5
        centroid_matrix = np.zeros((self.popSize, self.NVars))
        lam_seq = np.zeros(self.popSize)
        abTab = [(0.0, 0.0, 0.0) for _ in range(self.NVars)]
        centroids4dim = np.zeros(self.popSize)

        est_m          = np.zeros(self.NVars)
        mean_indiv     = np.zeros(self.NVars)
        mean_indivCl0  = np.zeros(self.NVars)
        mean_indivCl1  = np.zeros(self.NVars)
        mean_indiv_old = np.zeros(self.NVars)
        populLimCount = np.zeros(self.popSize, dtype=int)
        numOfStagIt = 0
        ITS_MODULO = 17
        fit_mean_old = 1e20    

        for curIndx in range(self.NInds):
            self.FitMass[curIndx] = test_func(self.Popul[curIndx], self.NVars)
            self.FindNSaveBest(curIndx == 0, curIndx)
            if(not globalbestinit or self.bestfit < globalbest):
                globalbest = self.bestfit
                globalbestinit = True
            # self.SaveBestValues(self.func_num, self.RunN, self.bestfit) #TODO- usunąć func_num?

        while True:
            minfit = self.FitMass[0]
            maxfit = self.FitMass[0]
            for i in range(self.NInds):
                self.FitMassCopy[i] = self.FitMass[i]
                self.Indexes[i] = i
                if(self.FitMass[i] >= maxfit):
                    maxfit = self.FitMass[i]
                if(self.FitMass[i] <= minfit):
                    minfit = self.FitMass[i]
            if(minfit != maxfit):
                pairs = list(zip(self.FitMassCopy, self.Indexes))
                pairs.sort(key=lambda x: x[0])
                self.FitMassCopy, self.Indexes = map(list, zip(*pairs))
                # qSort2int(self.FitMassCopy,self.Indexes,0,self.NInds-1);
            for i in range(self.NInds):
                for j in range(self.NInds):
                    if(i == self.Indexes[j]):
                        self.BackIndexes[i] = j
                        break

            for i in range(self.NInds):
                FitTemp3.append(math.exp(-(i)/self.NInds))
            index = np.random.choice(len(FitTemp3), p=np.array(FitTemp3)/sum(FitTemp3))
            psizeval = max(2.0,self.NInds*(0.2/(maxFes*NFEval+0.2)))
            CrossExponential = 0
            if(Random(0,1) < 0.5):
                CrossExponential = 1
            for curIndx in range(self.NInds):
                MemoryCurrentIndex = IntRandom(self.MemorySize)
                Cr = min(1.0,max(0.0,NormRand(self.MemoryCr[MemoryCurrentIndex],0.1)))
                while True:
                    F = CachyRand(self.MemoryF[MemoryCurrentIndex], 0.1)
                    if F > 0:
                        break
                self.FGenerated[curIndx] = min(F,1.0)
                self.CrGenerated[curIndx] = Cr
            # qSort1(CrGenerated,0,NInds-1);
            self.CrGenerated = np.sort(self.CrGenerated)
            iterBestFit = 1E18
            iterBestIndx=0

            for curIndx in range(self.NInds):
                self.Rands[0] = self.Indexes[IntRandom(psizeval)]
                for i in range(25):
                    if CheckGenerated(0, self.Rands, curIndx):
                        break
                    self.Rands[0] = self.Indexes[IntRandom(psizeval)]

                GenerateNextRandUnif(1,self.NInds,self.Rands,curIndx)
                if(Random(0,1) > ArchProbs or self.CurrentArchiveSize == 0):
                    probabilities = np.array(FitTemp3)
                    probabilities = probabilities / probabilities.sum()
                    self.Rands[2] = self.Indexes[np.random.choice(len(FitTemp3), p=probabilities)]
                    for i in range(25):
                        if CheckGenerated(0, self.Rands, curIndx):
                            break
                        probabilities = np.array(FitTemp3)
                        probabilities = probabilities / probabilities.sum()
                        self.Rands[2] = self.Indexes[np.random.choice(len(FitTemp3), p=probabilities)]
                    self.ArchUsages[curIndx] = 0
                else:
                    GenerateNextRandUnifOnlyArch(2,self.NInds,self.CurrentArchiveSize,self.Rands,curIndx)
                    self.ArchUsages[curIndx] = 1
                    
                for j in range(self.NVars):
                    self.Donor[j] = (self.Popul[curIndx][j] +
                        self.FGenerated[curIndx]*(self.GetValue(self.Rands[0],self.NInds,j) - self.Popul[curIndx][j]) +
                        self.FGenerated[curIndx]*(self.GetValue(self.Rands[1],self.NInds,j) -self.GetValue(self.Rands[2],self.NInds,j))
                        )
                WillCrossover = IntRandom(self.NVars)
                Cr = self.CrGenerated[int(self.BackIndexes[curIndx])]
                CrToUse = 0
                if(NFEval > 0.5*maxFes):
                    CrToUse = (NFEval/maxFes-0.5)*2
                if(CrossExponential == 0):
                    for j in range(self.NVars):
                        if(Random(0,1) < CrToUse or WillCrossover == j):
                            self.PopulTemp[curIndx][j] = self.Donor[j]
                        else:
                            self.PopulTemp[curIndx][j] = self.Popul[curIndx][j]
                else:
                    StartLoc = IntRandom(self.NVars)
                    L = StartLoc+1
                    while(Random(0,1) < Cr and L < self.NVars):
                        L+=1
                    for j in range(self.NVars):
                        self.PopulTemp[curIndx][j] = self.Popul[curIndx][j]
                    for j in range(StartLoc, L):
                        self.PopulTemp[curIndx][j] = self.Donor[j]

                FindLimits(self.PopulTemp[curIndx],self.Popul[curIndx],self.NVars,self.Left,self.Right)
                # self.FitMassTemp[curIndx] = cec_21_(self.PopulTemp[curIndx],self.func_num,self.bench_num);
                self.FitMass[curIndx] = test_func(self.PopulTemp[curIndx], self.NVars)
                if(self.FitMassTemp[curIndx] <= globalbest):
                    globalbest = self.FitMassTemp[curIndx]

                if(self.FitMassTemp[curIndx] < self.FitMass[curIndx]):
                    self.SaveSuccessCrF(Cr,F,abs(self.FitMass[curIndx]-self.FitMassTemp[curIndx]))
                self.FindNSaveBest(False,curIndx)
                # self.SaveBestValues(self.func_num,self.RunN,self.bestfit)
            ArchSuccess = 0
            NoArchSuccess = 0
            NArchUsages = 0
            for curIndx in range(self.NInds):
                if(self.FitMassTemp[curIndx] <= self.FitMass[curIndx]):
                    if(self.ArchUsages[curIndx] == 1):
                        ArchSuccess += (self.FitMass[curIndx] - self.FitMassTemp[curIndx])/self.FitMass[curIndx]
                        NArchUsages += 1
                    else:
                        NoArchSuccess+=(self.FitMass[curIndx] - self.FitMassTemp[curIndx])/self.FitMass[curIndx]
                    self.CopyToArchive(self.Popul[curIndx],self.FitMass[curIndx])
                    for j in range(self.NVars):
                        self.Popul[curIndx][j] = self.PopulTemp[curIndx][j]
                    self.FitMass[curIndx] = self.FitMassTemp[curIndx]

            if(NArchUsages != 0):
                ArchSuccess = ArchSuccess/NArchUsages
                NoArchSuccess = NoArchSuccess/(self.NInds-NArchUsages)
                ArchProbs = ArchSuccess/(ArchSuccess + NoArchSuccess)
                ArchProbs = max(0.1,min(0.9,ArchProbs))
                if(ArchSuccess == 0):
                    ArchProbs = 0.5
            else:
                ArchProbs = 0.5
            newNInds = round((self.NIndsMin-self.NIndsMax)*pow((NFEval/(maxFes)),(1.0-(NFEval)/(maxFes)))+self.NIndsMax)
            if(newNInds < self.NIndsMin):
                newNInds = self.NIndsMin
            if(newNInds > self.NIndsMax):
                newNInds = self.NIndsMax
            newArchSize = round((self.NIndsMin-self.NIndsMax)*pow(((NFEval)/(maxFes)),(1.0-(NFEval)/(maxFes)))+self.NIndsMax)*self.ArchiveSizeParam
            if(newArchSize < self.NIndsMin):
                newArchSize = self.NIndsMin
            ArchiveSize = newArchSize
            if(self.CurrentArchiveSize >= ArchiveSize):
                self.CurrentArchiveSize = ArchiveSize
            self.RemoveWorst(self.NInds,newNInds)
            self.NInds = newNInds
            self.UpdateMemoryCrF()
            self.SuccessFilled = 0
            self.Generation += 1

            #while warunke
            if (NFEval < maxFes):
                break





In [112]:
opt = Optimizer(newNInds=50, newNVars=dimensionality, new_func_num=1, newRunN=1,
                new_bench_num=0, NewMemSize=10, NewArchSizeParam=0.5)

opt.MainCycle()
print("Najlepsza wartość funkcji celu (bestfit):", opt.bestfit)
print("Najlepsze znalezione rozwiązanie (BestInd):", opt.BestInd)


Najlepsza wartość funkcji celu (bestfit): 17012.976567779682
Najlepsze znalezione rozwiązanie (BestInd): [-42.64324089  -0.53101753  38.3932339  -10.41349813  -2.03520057
 -82.51732129  54.85914013 -11.0139376   10.43031735 -59.65646059]


In [113]:
# def main():
#     f = 0.0
#     # Seeds = [0] * 1000
#     # with open("input_data/Rand_Seeds.txt", "r") as rs:
#     #     for i in range(1000):
#     #         f = float(rs.readline())
#     #         Seeds[i] = int(f)
#     Seeds = [random.randint(0, 2**31 - 1) for _ in range(1000)]


#     if dimensionality == 10:
#         maxFes = 200000
#     if dimensionality == 20:
#         maxFes = 1000000

#     POP_SIZE = dimensionality * 5
#     oldPopSize = POP_SIZE

#     for runNum in range(NMBR_OF_RUNS):
#         for func_num in range(1, 13):

#             seed_ind = (dimensionality // 10 * func_num * NMBR_OF_RUNS + runNum) - NMBR_OF_RUNS
#             seed_ind = seed_ind % 1000 + 1

#             seed1 = Seeds[seed_ind]
#             random.seed(seed1)

#             fopt = 0
#             if func_num == 1:
#                 fopt = 300
#             elif func_num == 2:
#                 fopt = 400
#             elif func_num == 3:
#                 fopt = 600
#             elif func_num == 4:
#                 fopt = 800
#             elif func_num == 5:
#                 fopt = 900
#             elif func_num == 6:
#                 fopt = 1800
#             elif func_num == 7:
#                 fopt = 2000
#             elif func_num == 8:
#                 fopt = 2200
#             elif func_num == 9:
#                 fopt = 2300
#             elif func_num == 10:
#                 fopt = 2400
#             elif func_num == 11:
#                 fopt = 2600
#             elif func_num == 12:
#                 fopt = 2700

#             globalBestFit = False
#             initfinished = False
#             LastFEcount = 0
#             NFEval = 0
#             bench_num = 0
            
#             OptZ = Optimizer(POP_SIZE, dimensionality, func_num, runNum, bench_num, 20 * dimensionality, 2.1)

#             # #if defined (SR_SAVE_AS_NEAREST_RESTART) || defined (K_MEANS_AS_NEAREST)
#             # OptZ.Initialize(POP_SIZE, dimensionality, func_num, runNum, 20 * dimensionality, 2.1)
#             isRestart = False

#             # while (globalBestFit - fopt > MIN_ERROR and NFEval < maxFes):
#             #     OptZ.MainCycle(fopt)

#             #     POP_SIZE = 400  # best

#             #     OptZ.restart(POP_SIZE, dimensionality, func_num, runNum, 20 * dimensionality, 2.1)
#             #     isRestart = True

#             POP_SIZE = oldPopSize
#             isRestart = False
#             # #endif

#             # OptZ.Clean()
#             if isinstance(globalBestFit, (int, float)):
#                 diff = globalBestFit - fopt
#             else:
#                 diff = "N/A"

#             print(f"{dimensionality} Run {runNum + 1} f {func_num} global best {diff} best sol:")
#             print()

#     for func_num in range(1, 13):
#         filename = "NL-SHADE"

#         buffer = f"{func_num}_{dimensionality}_pop:{POP_SIZE}.txt"
#         filename += buffer
#         print(filename)
#         with open(filename, "w") as fout:
#             for step in range(17):
#                 for runNum in range(NMBR_OF_RUNS):
#                     if step == 16 and ResultsArray[func_num - 1][runNum][step] == 0:
#                         ResultsArray[func_num - 1][runNum][step] = maxFes
#                     fout.write(str(ResultsArray[func_num - 1][runNum][step]) + "\t")
#                 fout.write("\n")


# if __name__ == "__main__":
#     main()
